# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook provides a step-by-step guide to loading and exploring the FAIR² dataset on second primary colorectal cancer in cancer survivors, using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is made available via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Let's load the dataset metadata with `mlcroissant`, and display its title and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata object as attributes
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's review the available record sets, their IDs, as well as fields and columns provided by this Croissant dataset definition. 

We use the `@id` (identifier) of each record set and field as required. This ensures robust referencing and data extraction.

In [ ]:
# List all record sets and their field @ids
record_sets = metadata.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field name: {field.name}")
        print(f"      @id: {field.id}")
        print(f"      Data type: {field.data_type}")
    print()

In summary:
- Use the record set `@id` to access its records.
- Use field `@id`s for selecting and analyzing data.

---

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. We will use the record set and field `@id` values shown above.

If there is only one main record set (as is often the case for clinical tabular datasets), we'll extract from it. Otherwise, adjust the list as needed.

In [ ]:
# Gather all available record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
print('Record set @ids:', record_set_ids)

# Extract each record set into a pandas DataFrame
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for '{record_set_id}' has shape: {df.shape}")

# Preview the first DataFrame
if record_set_ids:
    example_rs = record_set_ids[0]
    print(f"\nColumns for record set '{example_rs}':")
    print(dataframes[example_rs].columns.tolist())
    dataframes[example_rs].head()

## 4. Exploratory Data Analysis (EDA)

We'll apply some typical data processing steps:
- Filtering records based on a numeric field (e.g., Age).
- Normalizing the selected numeric field.
- Grouping data by a categorical field (e.g., Sex or tumor anatomical location).

For this dataset, suppose the record set field IDs for `Age` and `Sex` are available (replace with the correct `@id`s as shown in the overview above if different):

In [ ]:
# Choose the main record set (usually only one for tabular clinical data)
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Display the DataFrame columns again to help select fields
print("Available columns:", df.columns.tolist())

# Find plausible numeric field id for Age (try common variants):
candidate_age_fields = [col for col in df.columns if 'age' in col.lower()]
print("Candidate Age fields:", candidate_age_fields)
# Choose the first match for demonstration; adjust as needed
numeric_field_id = candidate_age_fields[0] if candidate_age_fields else df.columns[0]  # fallback to first column

# Find a plausible group field (e.g., sex, gender, tumor_site)
candidate_group_fields = [col for col in df.columns if any(x in col.lower() for x in ['sex', 'gender', 'site', 'location'])]
print("Candidate Group fields:", candidate_group_fields)
group_field_id = candidate_group_fields[0] if candidate_group_fields else df.columns[0]

# Filtering: select patients older than a threshold (e.g., 65 years)
threshold = 65
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
else:
    # Try converting to numeric if stored as string
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
    filtered_df[numeric_field_id] = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')

print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization
mean_val = filtered_df[numeric_field_id].mean()
std_val = filtered_df[numeric_field_id].std()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by group_field_id
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean','count','std'])
    print(f"\nGrouped data by {group_field_id}:")
    print(grouped_df.head())
else:
    print(f"\n{group_field_id} not found in columns for grouping.")

## 5. Visualization

Let's visualize the distribution of Ages and compare them by group (e.g., Sex or Tumor Location).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=12, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group
if group_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, you:
- Loaded the FAIR² clinical dataset with mlcroissant via the Croissant schema URL
- Discovered its record sets and fields by `@id`
- Extracted tabular records into pandas DataFrames
- Performed basic EDA: filtering, normalization, and group statistics
- Visualized a key clinical variable's distribution

You can extend this notebook to investigate further aspects, explore relationships between more fields, and design your own analyses.

*Remember to always refer to fields, record sets, and columns by their `@id` as required for maximum reproducibility and schema robustness!*